In [4]:
%pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from rapidfuzz import process, fuzz

import warnings
warnings.filterwarnings('ignore')


In [6]:
# Sample df1 with words to match
df_ACFull = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="Autocare")
df_ACFull

,PartTerminologyName,PA,Unnamed: 2
0,4WD Actuator,Terminal Gender,4WD ActuatorTerminal Gender
1,4WD Actuator,Terminal Quantity,4WD ActuatorTerminal Quantity
2,4WD Actuator,Mounting Hardware Included,4WD ActuatorMounting Hardware Included
3,4WD Actuator,Wiring Harness Length,4WD ActuatorWiring Harness Length
4,4WD Actuator,Connector Gender,4WD ActuatorConnector Gender
...,...,...,...
4738,Wire Holder,Wire Gauge Measurement,Wire HolderWire Gauge Measurement
4739,Wire Holder,Mounting Hardware Included,Wire HolderMounting Hardware Included
4740,Wire Holder,Insulated Coating,Wire HolderInsulated Coating
4741,Wire Holder,Wire Quantity,Wire HolderWire Quantity


In [7]:
# Sample df1 with words to match
df_FBGFull = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="FBG")
df_FBGFull=df_FBGFull[['Part Terminology Name','Cleaned Attributes']]

In [8]:
df_FBGFull.size

10088

In [9]:
PartNames=list(set(df_FBGFull['Part Terminology Name'].tolist()))

In [10]:
cols =['Part Terminology Name','Cleaned Attributes','approx_matches'] #,'Review_Mentions','Standards'
df_New = pd.DataFrame(columns=cols)
count=0

In [11]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 45

In [12]:
for i in range (len(PartNames)):
    df_FBGF=df_FBGFull[df_FBGFull['Part Terminology Name']==PartNames[i]].reset_index(drop=True)
    df_ACF=df_ACFull[df_ACFull['PartTerminologyName']==PartNames[i]]

    for j in range(len(df_FBGF)):
        #print(PartNames[i],df_FBGF['Cleaned Attributes'][j])
        df_New.at[count,'Part Terminology Name']=PartNames[i]
        df_New.at[count,'Cleaned Attributes']=df_FBGF['Cleaned Attributes'][j]
        df_New.at[count,'approx_matches']=get_matches(df_FBGF['Cleaned Attributes'][j],df_ACF['PA'].tolist(), threshold=MATCH_THRESHOLD)
        count=count+1  

In [13]:
df_New = df_New[df_New['approx_matches'].apply(str) != "[]"].reset_index(drop=True)
df_New = df_New.drop_duplicates(subset=['Part Terminology Name', 'Cleaned Attributes'])
df_New

,Part Terminology Name,Cleaned Attributes,approx_matches
0,Disc Brake Caliper Pin Boot Kit,Washers Included,"[Caliper Grease Included, Bolts Included, Pin ..."
1,Disc Brake Caliper Pin Boot Kit,Length (MM),"[Pin Length, Boot Length]"
2,Disc Brake Caliper Pin Boot Kit,Diameter (IN),"[Pin Inside Diameter, Pin Outside Diameter]"
3,Disc Brake Caliper Pin Boot Kit,Nuts Included,"[Bolts Included, Pin Included, Caliper Grease ..."
4,Disc Brake Caliper Pin Boot Kit,Sleeve Included,"[Bolts Included, Caliper Grease Included, Pin ..."
...,...,...,...
1214,Disc Brake Rotor and Hub Assembly,Salability Restrictions,[Construction]
1215,Disc Brake Rotor and Hub Assembly,Brake Rotor Material,"[Material, Brake Surface Finish, Rotor Outside..."
1216,Disc Brake Rotor and Hub Assembly,"Surface Finish, Disc","[Brake Surface Finish, Hat Finish, Disc Finish..."
1217,Disc Brake Rotor and Hub Assembly,OEM Vane Count,[Outer Bearing Cup]


In [15]:
df_New.to_excel(r'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\20250507_approximate_matches1.xlsx', index=False)

## Misc Code


In [ ]:

# Sample df1 with words to match
df_FB = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="FB")
df_FB

,Attributes,Cleaned FBG Attribute
0,(B03) Hazardous Material Code,Hazardous Material Code
1,(B05) Base Item ID,Base Item ID
2,(B29) VMRS Brand ID,VMRS Brand ID
3,(B32) Item Qty Size (Each),Item Qty Size (Each)
4,(B34) Container Type,Container Type
...,...,...
1269,Wire Quantity,Wire Quantity
1270,Wiring Harness Included,Wiring Harness Included
1271,Wiring Harness Length,Wiring Harness Length
1272,Wiring Harness Length (mm),Wiring Harness Length (mm)


In [ ]:
df_AC = pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Potential Match.xlsx",sheet_name="AC")
df_AC

,AC Attribute
0,Wiring Harness Length (mm)
1,Wiring Harness Length (in)
2,Height (mm)
3,Length (mm)
4,Width (mm)
...,...
1481,Linkage Frame or Bracket
1482,Rotation Direction
1483,Wiper Arm Included
1484,Insulated Coating


In [ ]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 80


In [ ]:

# Apply the function to df1
df_FB['approx_matches'] = df_FB['Cleaned FBG Attribute'].apply(
    lambda w: get_matches(w, df_AC['AC Attribute'].tolist(), threshold=MATCH_THRESHOLD)
)

df_FB=df_FB.drop_duplicates(inplace=True).rest_index(drop=True)


,Attributes,Cleaned FBG Attribute,approx_matches
0,(B03) Hazardous Material Code,Hazardous Material Code,[]
1,(B05) Base Item ID,Base Item ID,[]
2,(B29) VMRS Brand ID,VMRS Brand ID,[]
3,(B32) Item Qty Size (Each),Item Qty Size (Each),[]
4,(B34) Container Type,Container Type,[]
...,...,...,...
1269,Wire Quantity,Wire Quantity,[Wire Quantity]
1270,Wiring Harness Included,Wiring Harness Included,[Wiring Harness Included]
1271,Wiring Harness Length,Wiring Harness Length,"[Wiring Harness Length (m), Wiring Harness Len..."
1272,Wiring Harness Length (mm),Wiring Harness Length (mm),"[Wiring Harness Length (mm), Wiring Harness Le..."


In [ ]:
df_FB.to_excel(r'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\202approximate_matches.xlsx', index=False)